# Environment Setup

In [2]:
%env no_proxy='a.test.com,127.0.0.1,2.2.2.2'
!pip uninstall mindspore --y
!pip install mindspore==2.0.0 download nltk

env: no_proxy='a.test.com,127.0.0.1,2.2.2.2'
Looking in indexes: http://repo.myhuaweicloud.com/repository/pypi/simple/
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 947.2/947.2 MB 2.5 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 42.0 MB/s eta 0:00:0000:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 38.1/38.1 MB 54.7 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 761.6/761.6 kB 48.3 MB/s eta 0:00:00
  Attempting uninstall: scipy
    Found existing installation: scipy 1.5.2
    Uninstalling scipy-1.5.2:
      Successfully uninstalled scipy-1.5.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
mindvision 0.1.0 requires scikit-learn>=0.23.1, but you have scikit-learn 0.22.1 which is incompatible.
mindinsight 1.7.0 requires pyyaml>=5.3.1, but you have pyyaml 5.1 which is incompatible.
mindinsight 1.7.0 

In [3]:
# Check the current MindSpore version.
!pip show mindspore

Name: mindspore
Version: 2.0.0
Summary: MindSpore is a new open source deep learning training/inference framework that could be used for mobile, edge and cloud scenarios.
Home-page: https://www.mindspore.cn
Author: The MindSpore Authors
Author-email: contact@mindspore.cn
License: Apache 2.0
Location: /home/ma-user/anaconda3/envs/MindSpore/lib/python3.7/site-packages
Requires: asttokens, astunparse, numpy, packaging, pillow, protobuf, psutil, scipy
Required-by: 


In [4]:
import mindspore

mindspore.set_context(device_target="CPU")
print(mindspore.get_context(attr_key='device_target'))

CPU


In [5]:
from download import download
import re

url = "https://modelscope.cn/api/v1/datasets/SelinaRR/Multi30K/repo?Revision=master&FilePath=Multi30K.zip"

download(url, './', kind='zip', replace=True)

datasets_path = './datasets/'
train_path = datasets_path + 'train/'
valid_path = datasets_path + 'valid/'
test_path = datasets_path + 'test/'

def print_data(data_file_path, print_n=5):
    print("=" * 40 + "datasets in {}".format(data_file_path) + "=" * 40)
    with open(data_file_path, 'r', encoding='utf-8') as en_file:
        en = en_file.readlines()[:print_n]
        for index, seq in enumerate(en):
            print(index, seq.replace('\n', ''))


print_data(train_path + 'train.de')
print_data(train_path + 'train.en')


file_sizes: 1.37MB [00:01, 1.10MB/s]                                            
Extracting zip file...
Successfully downloaded / unzipped to ./
========================================datasets in ./datasets/train/train.de========================================
0 Zwei junge weiße Männer sind im Freien in der Nähe vieler Büsche.
1 Mehrere Männer mit Schutzhelmen bedienen ein Antriebsradsystem.
2 Ein kleines Mädchen klettert in ein Spielhaus aus Holz.
3 Ein Mann in einem blauen Hemd steht auf einer Leiter und putzt ein Fenster.
4 Zwei Männer stehen am Herd und bereiten Essen zu.
========================================datasets in ./datasets/train/train.en========================================
0 Two young, White males are outside near many bushes.
1 Several men in hard hats are operating a giant pulley system.
2 A little girl climbing into a wooden playhouse.
3 A man in a blue shirt is standing on a ladder cleaning a window.
4 Two men are at the stove preparing food.


In [6]:
import os
train_path_2k = datasets_path + 'train_2k/'
valid_path_100 = datasets_path + 'valid_100/'

def extract_first_n_lines(input_file, output_file, n_lines=2000):
    # Check whether the output directory exists.
    output_dir = os.path.dirname(output_file)
    if output_dir and not os.path.exists(output_dir):
        os.makedirs(output_dir)
    with open(input_file, "r", encoding="utf-8") as f_in:
        with open(output_file, "w", encoding="utf-8") as f_out:
            for i, line in enumerate(f_in):
                if i >= n_lines:
                    break  # Stop after 2,000 lines.
                f_out.write(line)
    print(f"The first {n_lines} lines have been extracted and saved to {output_file}")    
extract_first_n_lines(train_path+"train.en", train_path_2k+"train_2000.en", 2000)                
extract_first_n_lines(train_path+"train.de", train_path_2k+"train_2000.de", 2000)

extract_first_n_lines(valid_path+"val.en", valid_path_100+"val_100.en", 100)                
extract_first_n_lines(valid_path+"val.de", valid_path_100+"val_100.de", 100)

The first 2000 lines have been extracted and saved to ./datasets/train_2k/train_2000.en
The first 2000 lines have been extracted and saved to ./datasets/train_2k/train_2000.de
The first 100 lines have been extracted and saved to ./datasets/valid_100/val_100.en
The first 100 lines have been extracted and saved to ./datasets/valid_100/val_100.de


## **Task 1: Preprocessing data**

**Subtask 1: Tokenize text** Create a Multi30K data loader to split sentences into tokens, including words and punctuation. When you call this class to traverse the text, it returns the token lists for both the source language (German) and the target language (English) each time. 

1. Define the tokenize function in the _load method. This function uses re.findall to turn all word characters in the text to lowercase and return them. Set the pattern parameter in findall to r'\w+|[^\w\s]';

2. Process the English and German texts separately by tokenizing them into lists of lowercase words;

**Example**: "Hello world!" --&gt; ["hello", "world", "!"]

3. Use the Multi30K data loader to get the train_dataset, valid_dataset, and test_dataset. The training dataset has 2,000 text pairs, and the validation dataset has 100 text pairs;

In [7]:
class Multi30K():
    """Multi30K dataset loader
    
    Load the Multi30K dataset and process it as a Python iteration object. 
    
    """
    def __init__(self, path):
        self.data = self._load(path)
        
    def _load(self, path):
        # Question 1-1-1: Define the tokenize function in the _load method. This function uses re.findall to turn all word characters in the text to lowercase and return them,
        # Set the pattern parameter in findall to r'\w+|[^\w\s]';
        def tokenize(text):
            text = text.lower()
            return re.findall(r'\w+|[^\w\s]', text)
        
        members = {i.split('.')[-1]: i for i in os.listdir(path)}
        de_path = os.path.join(path, members['de'])
        en_path = os.path.join(path, members['en'])
        # Question 1-1-2: Process the English and German texts separately by tokenizing them into lists of lowercase words;
        with open(de_path, 'r', encoding='utf-8') as de_file:
            de = de_file.readlines()
            de = [tokenize(line) for line in de]
        with open(en_path, 'r', encoding='utf-8') as en_file:
            en = en_file.readlines()
            en =[tokenize(line) for line in en]

        return list(zip(de, en))
        
    def __getitem__(self, idx):
        return self.data[idx]
    
    def __len__(self):
        return len(self.data)

In [8]:
# Question 1-1-3: Use the Multi30K data loader to get the train_dataset, valid_dataset, and test_dataset;
train_dataset = Multi30K(train_path_2k)
valid_dataset = Multi30K(valid_path_100)
test_dataset = Multi30K(test_path)

Test the decompression and word segmentation results and print the English and German text of test dataset, where we can see that each word and punctuation have been separated. 

In [9]:
for de, en in test_dataset:
    print(f'de = {de}')
    print(f'en = {en}')
    break

de = ['ein', 'mann', 'mit', 'einem', 'orangefarbenen', 'hut', ',', 'der', 'etwas', 'anstarrt', '.']
en = ['a', 'man', 'in', 'an', 'orange', 'hat', 'starring', 'at', 'something', '.']


#### Vocabulary

Map each token to a numeric index starting from 0 to save space and remove less common tokens. The resulting set of tokens and numbers is called a vocabulary. 

The "Hello world!" example uses this vocabulary:


{"&lt;unk&gt;": 0, "&lt;pad&gt;": 1, "&lt;bos&gt;": 2, "&lt;eos&gt;": 3, "hello": 4, "world": 5, "!": 6}


Four special tokens are used in dictionary building. 

- &lt;unk&gt;: Unknown tokens are words that appear less than a set number of times;
- &lt;bos&gt;: Beginning of sentence tokens mark the start of a sentence;
- &lt;eos&gt;: End of sentence tokens mark the end of a sentence;
- &lt;pad&gt;: Padding tokens add padding when a sentence is too short;

After creating a vocabulary with Vocab, you can convert between tokens and numerical indices. You can call the encode function to get the numerical index or index sequence for the input token or token sequence. Similarly, you can call the decode function to get the token or token sequence for the input numerical index or index sequence. 

**Subtask 2: Build a vocabulary**

1. Map numeric indices to words;

2. Map a word or word array to a numeric index or numeric index array;

3. Map a single numerical index or numerical index array to a single word or word array;


In [99]:
class Vocab:
    """Build a vocabulary based on the word frequency dictionary."""

    special_tokens = ['<unk>', '<pad>', '<bos>', '<eos>']

    def __init__(self, word_count_dict, min_freq=1):
        self.word2idx = {}
        for idx, tok in enumerate(self.special_tokens):
            self.word2idx[tok] = idx

        filted_dict = {
            w: c
            for w, c in word_count_dict.items() if c >= min_freq
        }
        for w, _ in filted_dict.items():
            self.word2idx[w] = len(self.word2idx)

        self.idx2word = {idx: word for word, idx in self.word2idx.items()}

        self.bos_idx = self.word2idx['<bos>']
        self.eos_idx = self.word2idx['<eos>']
        self.pad_idx = self.word2idx['<pad>']
        self.unk_idx = self.word2idx['<unk>']

    def _word2idx(self, word):
        if word not in self.word2idx:
            return self.unk_idx
        return self.word2idx[word]

    def _idx2word(self, idx):
        # Question 1-2-1
        if idx not in self.idx2word:
            raise ValueError('input index is not in vocabulary.')
        return self.idx2word[idx]

    def encode(self, word_or_list):
        # Question 1-2-2
        if isinstance(word_or_list, list):
            return [self._word2idx(w) for w in word_or_list]
        return self._word2idx(word_or_list)

    def decode(self, idx_or_list):
        # Question 1-2-3
        if isinstance(idx_or_list, list):
            return [self._idx2word(i) for i in idx_or_list]
        return self._idx2word(idx_or_list)

    def __len__(self):
        return len(self.word2idx)

In [12]:
word_count = {'a':20, 'b':10, 'c':1, 'd':2}

vocab = Vocab(word_count, min_freq=2)
len(vocab)

7

In [13]:
from collections import Counter, OrderedDict

def build_vocab(dataset):
    de_words, en_words = [], []
    for de, en in dataset:
        de_words.extend(de)
        en_words.extend(en)

    de_count_dict = OrderedDict(sorted(Counter(de_words).items(), key=lambda t: t[1], reverse=True))
    en_count_dict = OrderedDict(sorted(Counter(en_words).items(), key=lambda t: t[1], reverse=True))

    return Vocab(de_count_dict, min_freq=2), Vocab(en_count_dict, min_freq=2)

In [14]:
de_vocab, en_vocab = build_vocab(train_dataset)
print('Unique tokens in de vocabulary:', len(de_vocab))

Unique tokens in de vocabulary: 1288


#### Data Iterator

The final step in data preprocessing is to create a data iterator. You have turned the German and English text descriptions into token sequences using the Multi30K data loader and created a vocabulary that links tokens to numbers. Next, you need to convert the token sequence into a numerical index sequence. 

The following still uses "Hello world!" as an example to show the step-by-step operations in the data iterator.

1. Add the special tokens &lt;bos&gt; at the start and &lt;eos&gt; at the end of each token sequence. 


["hello", "world", "!"] --&gt; ["&lt;bos&gt;", "hello", "world", "!", "&lt;eos&gt;"]


2. Make all sequences the same length by cutting off extra parts or adding padding where needed using &lt;pad&gt;. Record each sequence's valid length. Assume that the unified length is 7. 


["&lt;bos&gt;", "hello", "world", "!", "&lt;eos&gt;"] --&gt; ["&lt;bos&gt;", "hello", "world", "!", "&lt;eos&gt;", "&lt;pad&gt;", "&lt;pad&gt;"], valid length = 5


3. Finally, perform batch processing on the text sequences. For each batch, use the encode function to get the numeric index for all sequence tokens, and then obtain the results as a tensor. 


["&lt;bos&gt;", "hello", "world", "!", "&lt;eos&gt;", "&lt;pad&gt;", "&lt;pad&gt;"] --&gt; [2, 4, 5, 6, 3, 1, 1] --&gt; tensor

**Subtask 3: Create a data iterator**

1. Add the special tokens &lt;bos&gt; at the start and &lt;eos&gt; at the end of each token sequence;

2. Make all sequences the same length by cutting off extra parts or adding padding where needed using &lt;pad&gt;. Record each sequence's valid length;

3. For each batch, use the encode function to get the numeric index for all sequence tokens, and then make all sequences the same length;

4. The function returns src_idx, src_len, and trg_idx as a tensor with the data type INT32;

In [15]:
import mindspore

class Iterator():
    """Create a data iterator."""
    def __init__(self, dataset, de_vocab, en_vocab, batch_size, max_len=32, drop_reminder=False):
        self.dataset = dataset
        self.de_vocab = de_vocab
        self.en_vocab = en_vocab

        self.batch_size = batch_size
        self.max_len = max_len
        self.drop_reminder = drop_reminder

        length = len(self.dataset) // batch_size
        self.len = length if drop_reminder else length + 1

    def __call__(self):
        def pad(idx_list, vocab, max_len):
            """Unify the sequence length and record the valid length."""
            idx_pad_list, idx_len = [], []
            # Question 1-3-1: Add the special tokens <bos> at the start and <eos> at the end of each token sequence. 
            # Question 1-3-2: Make all sequences the same length by cutting off extra parts or adding padding where needed using <pad>. Record each sequence's valid length. 
            for i in idx_list:
                # Add the following code:
                i=[vocab.bos_idx]+list(i)+[vocab.eos_idx]
                valid_len=min(len(i), max_len)
                idx_len.append(valid_len)
                if len(i)>= max_len:
                       i = i[:max_len]
                else:
                    i=i+[vocab.pad_idx]*(max_len-len(i))
                idx_pad_list.append(i)
               
            return idx_pad_list, idx_len

        def sort_by_length(src, trg):
            """Sort German/English field lengths."""
            data = zip(src, trg)
            data = sorted(data, key=lambda t: len(t[0]), reverse=True)
            return zip(*list(data))

        def encode_and_pad(batch_data, max_len):
            """Convert text data in batches into numeric indices and unify the length of each sequence."""
            # Question 1-3-3: For each batch, use the encode function to get the numeric index for all sequence tokens, and then make all sequences the same length;
            src_data, trg_data = zip(*batch_data)
            src_idx = [de_vocab.encode(list(s)) for s in src_data]
            trg_idx = [en_vocab.encode(list(t)) for t in trg_data]

            src_idx, trg_idx = sort_by_length(src_idx, trg_idx)
            src_idx_pad, src_len = pad(src_idx, de_vocab, max_len)
            trg_idx_pad, _ = pad(trg_idx, en_vocab, max_len)

            return src_idx_pad, src_len, trg_idx_pad

        for i in range(self.len):
            if i == self.len - 1 and not self.drop_reminder:
                batch_data = self.dataset[i * self.batch_size:]
            else:
                batch_data = self.dataset[i * self.batch_size: (i+1) * self.batch_size]

            src_idx, src_len, trg_idx = encode_and_pad(batch_data, self.max_len)
            # Questions 1-3-4: The function returns src_idx, src_len, and trg_idx as a tensor with the data type INT32;
            yield mindspore.Tensor(src_idx, mindspore.int32), mindspore.Tensor(src_len, mindspore.int32), mindspore.Tensor(trg_idx, mindspore.int32)

    def __len__(self):
        return self.len

In [17]:
train_iterator = Iterator(train_dataset, de_vocab, en_vocab, batch_size=128, max_len=32, drop_reminder=True)
valid_iterator = Iterator(valid_dataset, de_vocab, en_vocab, batch_size=128, max_len=32, drop_reminder=False)
test_iterator = Iterator(test_dataset, de_vocab, en_vocab, batch_size=1, max_len=32, drop_reminder=False)


for src_idx, src_len, trg_idx in train_iterator():
    print(f'src_idx.shape:{src_idx.shape}\n{src_idx}\nsrc_len.shape:{src_len.shape}\n{src_len}\ntrg_idx.shape:{trg_idx.shape}\n{trg_idx}')
    break

src_idx.shape:(128, 32)
[[ 2  5 13 ...  1  1  1]
 [ 2  5 13 ...  1  1  1]
 [ 2  5 13 ...  1  1  1]
 ...
 [ 2  5 51 ...  1  1  1]
 [ 2  9 49 ...  1  1  1]
 [ 2  5 28 ...  1  1  1]]
src_len.shape:(128,)
[27 25 24 24 23 23 23 23 22 22 22 21 21 21 21 21 20 20 20 20 20 19 19 19
 18 18 18 18 18 18 18 18 17 17 17 17 17 17 17 17 17 17 16 16 16 16 16 16
 16 16 16 16 15 15 15 15 15 15 15 15 15 15 15 14 14 14 14 14 14 14 14 14
 14 14 14 14 13 13 13 13 13 13 13 13 13 12 12 12 12 12 12 12 12 12 12 12
 12 12 12 12 12 12 12 12 11 11 11 11 11 11 11 11 11 11 10 10 10 10 10 10
 10  9  9  9  9  9  9  8]
trg_idx.shape:(128, 32)
[[  2   4 896 ...   1   1   1]
 [  2   4   9 ...   1   1   1]
 [  2   4   9 ...   1   1   1]
 ...
 [  2   4  57 ...   1   1   1]
 [  2   4  45 ...   1   1   1]
 [  2   4  31 ...   1   1   1]]


## **Task 2: Building a model**

**Subtask 1: Compute the attention score**

**Note:**

When computing the attention score, you mainly consider three factors: **query**, **key**, and **value**. 

- `query`: Task content
- `key`: index/tag (to help locate the answer)
- `value`: Answer

In text translation, the goal is for the translated sentence to convey the same meaning as the original. During attention score computation, the query usually is related to the translated sentence, while the key is related to the original sentence. 

The attention score denotes the similarity between the query and the key. This exam uses the scaled dot-product attention method to compute the attention score. To find out how similar the query and key are, calculate their dot product. 

To prevent the size of the query ($Q \in R^{n\times d_{model}}$) and key ($K \in R^{m\times d_{model}}$) from affecting the similarity calculation, divide their dot product by $\sqrt{d_{model}}$. 

$$\text{Attention Score}(Q, K)=\frac{QK^T}{\sqrt{d_{model}}}$$

Limit the similarity range to 0 to 1 and apply it to the value. 

$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_{model}}}\right)V$$

In this code, you will do the scaled dot-product attention calculation. When the class is called, it returns the weighted value (output) and attention weights (attn). 

**Complete the following code based on the preceding instructions:**

1. Obtain $\sqrt{d_{model}}$ based on the value of the last dimension of the obtained query, assign the value to the scaling_factor variable, and use mstype to specify the data type of scaling_factor as float32.

2. Calculate the similarity based on the formula: $$\text{Attention Score}(Q, K)=\frac{QK^T}{\sqrt{d_{model}}}$$ and assign the similarity to the attn variable.

3. Limit the similarity range to 0 to 1 using the formula ($$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_{model}}}\right)V$$). After dropout, apply the similarity to the value.



In [62]:
import mindspore
from mindspore import nn
from mindspore import ops
from mindspore import Tensor
from mindspore import dtype as mstype


class ScaledDotProductAttention(nn.Cell):
    def __init__(self, dropout_p=0.):
        super().__init__()
        self.softmax = nn.Softmax()
        self.dropout = nn.Dropout(p=dropout_p)
        self.sqrt = ops.Sqrt()


    def construct(self, query, key, value, attn_mask=None):
        """scaled dot product attention"""
        # Question 2-1-1: Define the scaling_factor variable;
        embed_size = query.shape[-1]
        scaling_factor = self.sqrt(Tensor(embed_size,mstype.float32 ))
        # Question 2-1-2: Define the attn variable based on the formula in "2";
        attn = ops.bmm(query, key.transpose(0, 1, 3, 2)) / scaling_factor


        if attn_mask is not None:
            attn = attn.masked_fill(attn_mask, -1e9)
        # Question 2-1-3: Calculate the attention score based on the formula and prompt in "3";
        attn = self.softmax(attn)
        attn = self.dropout(attn)
        output = ops.bmm(attn, value)
        return (output, attn)

In [63]:
attention = ScaledDotProductAttention()
q_s = k_s = v_s = ops.ones((128, 8, 32, 64), mindspore.float32)
attn_mask = ops.ones((128, 8, 32, 32), mindspore.float32)
attn_mask = mindspore.ops.gt(attn_mask, attn_mask)
output, attn = attention(q_s, k_s, v_s, attn_mask)
print(output.shape, attn.shape)

(128, 8, 32, 64) (128, 8, 32, 32)


In [64]:
def get_attn_subsequent_mask(seq_q, seq_k):
    batch_size, len_q = seq_q.shape
    batch_size, len_k = seq_k.shape

    ones = mnp.ones((batch_size, len_q, len_k), dtype=mstype.float32)
    subsequent_mask = mnp.triu(ones, k=1)  # keep as float32, NOT .astype(bool_)
    return subsequent_mask

In [65]:
q = k = Tensor([[1, 1, 0, 0]], mstype.float32)
pad_idx = 0
mask = get_attn_pad_mask(q, k, pad_idx)
print(mask)
print(q.shape, mask.shape)

[[[False False  True  True]
  [False False  True  True]
  [False False  True  True]
  [False False  True  True]]]
(1, 4) (1, 4, 4)


**Subtask 2: Define the MultiheadAttention class**

1. Initialize self.W_Q, self.W_K, self.W_V, and self.W_O using the nn.Dense API from the MindSpore framework with the d_model, d_k, and n_heads variables;

2. Calculate Q, K, and V using the initial weight data. Then, reshape them to (batch_size, -1, n_heads, d_k);

3. Use the scaled dot-product attention class to calculate the weighted value (context) and attention weights (attn). Then, calculate the model's output;

In [66]:
class MultiHeadAttention(nn.Cell):
    def __init__(self, d_model, d_k, n_heads, dropout_p=0.):
        super().__init__()
        self.n_heads = n_heads
        self.d_k = d_k
        # Question 2-2-1
        self.W_Q = nn.Dense(d_model, d_k * n_heads)
        self.W_K = nn.Dense(d_model, d_k * n_heads)
        self.W_V = nn.Dense(d_model, d_k * n_heads)
        self.W_O = nn.Dense(d_k * n_heads, d_model)
        self.attention = ScaledDotProductAttention(dropout_p=dropout_p)

    def construct(self, query, key, value, attn_mask):
        """
        query: [batch_size, len_q, d_model]
        key: [batch_size, len_k, d_model]
        value: [batch_size, len_k, d_model]
        attn_mask: [batch_size, seq_len, seq_len]
        """

        batch_size = query.shape[0]
        # Question 2-2-2
        q_s = self.W_Q(query).view((batch_size, -1, self.n_heads, self.d_k))
        k_s = self.W_K(key).view((batch_size, -1, self.n_heads, self.d_k))
        v_s = self.W_V(value).view((batch_size, -1, self.n_heads, self.d_k))

        q_s = q_s.transpose((0, 2, 1, 3))
        k_s = k_s.transpose((0, 2, 1, 3))
        v_s = v_s.transpose((0, 2, 1, 3))

        attn_mask = attn_mask.expand_dims(1)
        attn_mask = ops.tile(attn_mask, (1, self.n_heads, 1, 1))
        # Question 2-2-3
        context, attn = self.attention(q_s, k_s, v_s, attn_mask)

        context = context.transpose((0, 2, 1, 3)).view((batch_size, -1, self.n_heads * self.d_k))

        output = self.W_O(context)

        return output, attn

In [67]:
dmodel, dk, nheads = 10, 2, 5
q = k = v = ops.ones((1, 2, 10), mstype.float32)
attn_mask = Tensor([False]).broadcast_to((1, 2, 2))
multi_head_attn = MultiHeadAttention(dmodel, dk, nheads)
output, attn = multi_head_attn(q, k, v, attn_mask)
print(output.shape, attn.shape)

(1, 2, 10) (1, 5, 2, 2)


## Transformer Structure

The figure below shows the transformer's structure. Before entering the encoder or decoder, the source and target sequences must be processed. 

1. Word embedding turns a sequence into a word vector that a model can understand. This vector holds the sequence's **content information**. 
2. **Positional encoding** adds location details based on the content. 

<div align=center><img src="https://openi.pcl.ac.cn/mindspore-courses/step_into_llm/raw/branch/master/Season1.step_into_chatgpt/1.Transformer/assets/transformer_structure.png"></div>

In [68]:
from mindspore import numpy as mnp

class PositionalEncoding(nn.Cell):
    """Positional encoding"""

    def __init__(self, d_model, dropout_p=0.1, max_len=100):
        super().__init__()
        self.dropout = nn.Dropout(p = dropout_p)

        self.pe = ops.Zeros()((max_len, d_model), mstype.float32)
        pos = mnp.arange(0, max_len, dtype=mstype.float32).view((-1, 1))
        angle = ops.pow(10000.0, mnp.arange(0, d_model, 2, dtype=mstype.float32)/d_model)

        self.pe[:, 0::2] = ops.sin(pos/angle)
        self.pe[:, 1::2] = ops.cos(pos/angle)

    def construct(self, x):
        batch_size = x.shape[0]

        pe = self.pe.expand_dims(0)
        pe = ops.broadcast_to(pe, (batch_size, -1, -1))
        x = x + pe[:, :x.shape[1], :]
        return self.dropout(x)

In [69]:
x = ops.Zeros()((1, 2, 4), mstype.float32)
pe = PositionalEncoding(4)
print(pe(x))

[[[0.         1.         0.         1.        ]
  [0.84147096 0.5403023  0.00999983 0.99995   ]]]


**Subtask 3: Complete the code for the position-wise feedforward neural network PoswiseFeedForward class****

A position-wise feedforward neural network performs a non-linear transformation on each position in the input. It consists of two linear layers, and the ReLU activation function is required between the layers. 

$$\mathrm{FFN}(x) = \mathrm{ReLU}(xW_1 + b_1)W_2 + b_2$$
Define the PoswiseFeedForward class based on the preceding formula.

- Note:
- (1) d_model is the input and output dimensions of the PoswiseFeedForward class;
- (2) After being activated by ReLU, the input x needs to be dropped out before entering the next linear layer.

In [70]:
# Question 2-3-1: Complete the code for the PoswiseFeedForward class based on the given instructions.
class PoswiseFeedForward(nn.Cell):
    def __init__(self, d_ff, d_model, dropout_p=0.):
        super().__init__()
        self.fc1=nn.Dense(d_model, d_ff)
        self.fc2=nn.Dense(d_ff, d_model)
        self.relu=nn.ReLU()
        self.dropout=nn.Dropout(p=dropout_p)
        

    def construct(self, x):
        """Feedforward neural network
        x: [batch_size, seq_len, d_model]
        """
        output = self.fc1(x)
        output = self.relu(output)
        output = self.dropout(output)
        output = self.fc2(output)
        return output
 

In [71]:
x = ops.ones((1, 2, 4), mstype.float32)
ffn = PoswiseFeedForward(16, 4)
print(ffn(x).shape)

(1, 2, 4)


In [72]:
class AddNorm(nn.Cell):
    def __init__(self, d_model, dropout_p=0.):
        super().__init__()
        self.layer_norm = nn.LayerNorm((d_model, ), epsilon=1e-5)
        self.dropout = nn.Dropout(p=dropout_p)
    
    def construct(self, x, residual):
        return self.layer_norm(self.dropout(x) + residual)

In [73]:
x = ops.ones((1, 2, 4), mstype.float32)
residual = ops.ones((1, 2, 4), mstype.float32)
add_norm = AddNorm(4)
print(add_norm(x, residual).shape)

(1, 2, 4)


The transformer's **encoder** processes input sequences and turns them into context vectors. 

<div align=center><img src="https://openi.pcl.ac.cn/mindspore-courses/step_into_llm/raw/branch/master/Season1.step_into_chatgpt/1.Transformer/assets/encoder.png" width="400" height="600" alt="encoder"></div>

**Subtask 4: Complete the code for the EncoderLayer and Encoder classes**

1. In the __init__ method of EncoderLayer, initialize the multi-head attention layer self.enc_self_attn, the position-wise feedforward neural network self.pos_ffn, and two Add & Norm layers self.add_norm1 and self.add_norm2;

2. Complete the implementation of a single encoder layer in the construct method of EncoderLayer based on the previous figure;

3. In the Encoder class, use the nn.CellList method to stack the EncoderLayer class n_layers times. Then, initialize the word embedding and positional encoding;

In [74]:
class EncoderLayer(nn.Cell):
    def __init__(self, d_model, n_heads, d_ff, dropout_p=0.):
        super().__init__()
        d_k = d_model // n_heads
        if d_k * n_heads != d_model:
            raise ValueError(f"The `d_model` {d_model} can not be divisible by `num_heads` {n_heads}.")
        # Question 2-4-1: In the __init__ method of EncoderLayer, initialize the multi-head attention layer self.enc_self_attn,
        #           the position-wise feedforward neural network self.pos_ffn, and two Add & Norm layers self.add_norm1 and self.add_norm2;
        self.enc_self_attn = MultiHeadAttention(d_model, d_k, n_heads, dropout_p)
        self.pos_ffn = PoswiseFeedForward(d_ff, d_model, dropout_p)
        self.add_norm1 = AddNorm(d_model, dropout_p)
        self.add_norm2 = AddNorm(d_model, dropout_p)
        
    def construct(self, enc_inputs, enc_self_attn_mask):
        """
        enc_inputs: [batch_size, src_len, d_model]
        enc_self_attn_mask: [batch_size, src_len, src_len]
        """
        # Question 2-4-2: Complete the implementation of a single encoder layer in the construct method of EncoderLayer based on the previous figure;
        residual = enc_inputs

        enc_outputs, attn = self.enc_self_attn(enc_inputs, enc_inputs, enc_inputs, enc_self_attn_mask)

        enc_outputs = self.add_norm1(enc_outputs, residual)
        residual = enc_outputs

        enc_outputs = self.pos_ffn(enc_outputs)

        enc_outputs = self.add_norm2(enc_outputs, residual)


        return enc_outputs, attn

In [75]:
x = ops.ones((1, 2, 8), mstype.float32)
mask = Tensor([False]).broadcast_to((1, 2, 2))
encoder_layer = EncoderLayer(8, 4, 16)
output, attn = encoder_layer(x, mask)
print(output.shape, attn.shape)

(1, 2, 8) (1, 4, 2, 2)


In [76]:
class Encoder(nn.Cell):
    def __init__(self, src_vocab_size, d_model, n_heads, d_ff, n_layers, dropout_p=0.):
        super().__init__()
        # Question 2-4-3: In the Encoder class, use the nn.CellList method to stack the EncoderLayer class n_layers times. Then, initialize the word embedding and positional encoding;
        self.src_emb = nn.Embedding(src_vocab_size, d_model)
        self.pos_emb = PositionalEncoding(d_model, dropout_p)
        self.layers = nn.CellList([EncoderLayer(d_model, n_heads, d_ff, dropout_p) for _ in range(n_layers)])
        self.scaling_factor = ops.Sqrt()(Tensor(d_model, mstype.float32))

        
    def construct(self, enc_inputs, src_pad_idx):
        """enc_inputs : [batch_size, src_len]
        """
        enc_outputs = self.src_emb(enc_inputs.astype(mstype.int32))
        enc_outputs = self.pos_emb(enc_outputs * self.scaling_factor)

        enc_self_attn_mask = get_attn_pad_mask(enc_inputs, enc_inputs, src_pad_idx)

        enc_self_attns = []
        for layer in self.layers:
            enc_outputs, enc_self_attn = layer(enc_outputs, enc_self_attn_mask)
            enc_self_attns.append(enc_self_attn)
        return enc_outputs, enc_self_attns

**The decoder** turns the encoder's output into a predicted target sequence $\hat{Y}$. This prediction is then compared to the actual target output, $Y$, to calculate the loss during model training. 

<div align=center><img src="https://openi.pcl.ac.cn/mindspore-courses/step_into_llm/raw/branch/master/Season1.step_into_chatgpt/1.Transformer/assets/decoder.png" alt="decoder"></div>



Unlike the encoder, each decoder layer is composed of two layers of a multi-head attention mechanism, followed by an additional linear layer that outputs the predicted result for the target sequence. 

- Layer 1: **masked multi-head self-attention** for computing the attention score of the target sequence;
- Layer 2: used to compute the mapping between the context sequence and the target sequence. The output of the decoder's masked multi-head self-attention is used as the query, and the output (context sequence) of the encoder is used as the key and value;

**Subtask 5: Complete the code for the DecoderLayer and Decoder classes**

1. Build the get_attn_subsequent_mask function as prompted.
,
**Note:**

The model can only see the words up to time "t – 1" when processing the target sequence at time t. It should not receive any future words as input. 

To use only "t – 1" tokens for calculating multi-head attention scores at time t, add a time mask in the first multi-head attention. This mask reveals the words in the target sequence one at a time. 

Use a triangular matrix for the attention mask by calling mindspore.numpy.triu with k=1. Set the data type to mindspore.float32. Words above the diagonal are ignored in attention calculations and marked as 1. 

$$\begin{matrix}
0 & 1 & 1 & 1 & 1\\
0 & 0 & 1 & 1 & 1\\
0 & 0 & 0 & 1 & 1\\
0 & 0 & 0 & 0 & 1\\
0 & 0 & 0 & 0 & 0\\
\end{matrix}$$

The mask is generally referred to as a subsequent mask. 

2. In the __init__ method of DecoderLayer, initialize self.dec_self_attn for the decoder's self-attention, self.dec_enc_attn for the encoder-decoder's attention, self.pos_ffn for the feedforward neural network, and the three Add & Norm layers (self.add_norm1, self.add_norm2, self.add_norm3);

3. Complete the implementation of a decoder layer in the construct method of DecoderLayer based on the previous figure;

4. In the Decoder class, use the nn.CellList method to stack the DecoderLayer class n_layers times. Then, initialize the word embedding and positional encoding;

In [77]:
# Question 2-5-1: Construct the get_attn_subsequent_mask function.
def get_attn_subsequent_mask(seq_q, seq_k):
    """Generate a time mask, so that the decoder can only see the first "t – 1" elements of the sequence at time t.
    
    Args:
        seq_q (Tensor): query sequence, shape = [batch size, len_q]
        seq_k (Tensor): key sequence, shape = [batch size, len_k]
    """
    batch_size, len_q = seq_q.shape
    batch_size, len_k = seq_k.shape

    ones = mnp.ones((batch_size, len_q, len_k), dtype=mstype.float32)
    subsequent_mask = mnp.triu(ones, k=1).astype(mstype.bool_)
    return subsequent_mask

In [78]:
q = k = ops.ones((1, 4), mstype.float32)
mask = get_attn_subsequent_mask(q, k)
print(mask)

[[[False  True  True  True]
  [False False  True  True]
  [False False False  True]
  [False False False False]]]


In [79]:
class DecoderLayer(nn.Cell):
    def __init__(self, d_model, n_heads, d_ff, dropout_p=0.):
        super().__init__()
        d_k = d_model // n_heads
        if d_k * n_heads != d_model:
            raise ValueError(f"The `d_model` {d_model} can not be divisible by `num_heads` {n_heads}.")
        # Question 2-5-2: In the __init__ method of DecoderLayer, initialize self.dec_self_attn for the decoder's self-attention,
        #            self.dec_enc_attn for the encoder-decoder's attention, self.pos_ffn for the feedforward neural network,
        #            and the three Add & Norm layers (self.add_norm1, self.add_norm2, and self.add_norm3).
        self.dec_self_attn = MultiHeadAttention(d_model, d_k, n_heads, dropout_p)
        self.dec_enc_attn = MultiHeadAttention(d_model, d_k, n_heads, dropout_p)
        self.pos_ffn = PoswiseFeedForward(d_ff, d_model, dropout_p)
        self.add_norm1 = AddNorm(d_model, dropout_p)
        self.add_norm2 = AddNorm(d_model, dropout_p)
        self.add_norm3 = AddNorm(d_model, dropout_p)
        
    def construct(self, dec_inputs, enc_outputs, dec_self_attn_mask, dec_enc_attn_mask):
        """
        dec_inputs: [batch_size, trg_len, d_model]
        enc_outputs: [batch_size, src_len, d_model]
        dec_self_attn_mask: [batch_size, trg_len, trg_len]
        dec_enc_attn_mask: [batch_size, trg_len, src_len]
        """
        # Question 2-5-3: Complete the implementation of a decoder layer in the construct method of DecoderLayer based on the previous figure;
        residual = dec_inputs

        dec_outputs, dec_self_attn = self.dec_self_attn(dec_inputs, dec_inputs, dec_inputs, dec_self_attn_mask)

        dec_outputs = self.add_norm1(dec_outputs, residual)
        residual = dec_outputs
   
        dec_outputs, dec_enc_attn = self.dec_enc_attn(dec_outputs, enc_outputs, enc_outputs, dec_enc_attn_mask)

        dec_outputs = self.add_norm2(dec_outputs, residual)
        residual = dec_outputs

        dec_outputs = self.pos_ffn(dec_outputs)

        dec_outputs = self.add_norm3(dec_outputs, residual)

        return dec_outputs, dec_self_attn, dec_enc_attn

In [80]:
x = y = ops.ones((1, 2, 4), mstype.float32)
mask1 = mask2 = Tensor([False]).broadcast_to((1, 2, 2))
decoder_layer = DecoderLayer(4, 1, 16)
output, attn1, attn2 = decoder_layer(x, y, mask1, mask2)
print(output.shape, attn1.shape, attn2.shape)

(1, 2, 4) (1, 1, 2, 2) (1, 1, 2, 2)


In [81]:
class Decoder(nn.Cell):
    def __init__(self, trg_vocab_size, d_model, n_heads, d_ff, n_layers, dropout_p=0.):
        super().__init__()
        # Question 2-5-4:  In the Decoder class, use the nn.CellList method to stack the DecoderLayer class n_layers times. Then, initialize the word embedding and positional encoding;
        self.trg_emb = nn.Embedding(trg_vocab_size, d_model)
        self.pos_emb = PositionalEncoding(d_model, dropout_p)
        self.layers = nn.CellList([DecoderLayer(d_model, n_heads, d_ff, dropout_p) for _ in range(n_layers)])
        self.projection = nn.Dense(d_model, trg_vocab_size)
        self.scaling_factor = ops.Sqrt()(Tensor(d_model, mstype.float32))      
        
    def construct(self, dec_inputs, enc_inputs, enc_outputs, src_pad_idx, trg_pad_idx):
        """
        dec_inputs: [batch_size, trg_len]
        enc_inputs: [batch_size, src_len]
        enc_outputs: [batch_size, src_len, d_model]
        """
        dec_outputs = self.trg_emb(dec_inputs.astype(mstype.int32))
        dec_outputs = self.pos_emb(dec_outputs * self.scaling_factor)

        dec_self_attn_pad_mask = get_attn_pad_mask(dec_inputs, dec_inputs, trg_pad_idx)
        dec_self_attn_subsequent_mask = get_attn_subsequent_mask(dec_inputs, dec_inputs)
       # AFTER (fixed):
        dec_self_attn_mask = ops.gt(
            dec_self_attn_pad_mask.astype(mstype.float32) + dec_self_attn_subsequent_mask.astype(mstype.float32),
            0
        )

        dec_enc_attn_mask = get_attn_pad_mask(dec_inputs, enc_inputs, src_pad_idx)

        dec_self_attns, dec_enc_attns = [], []
        for layer in self.layers:
            dec_outputs, dec_self_attn, dec_enc_attn = layer(dec_outputs, enc_outputs, dec_self_attn_mask, dec_enc_attn_mask)
            dec_self_attns.append(dec_self_attn)
            dec_enc_attns.append(dec_enc_attn)

        dec_outputs = self.projection(dec_outputs)
        return dec_outputs, dec_self_attns, dec_enc_attns

**Subtask 6: Construct and instantiate a transformer**

1. Complete the code in the Transformer class based on the defined Encoder and Decoder classes;

2. Instantiate the transformer model based on the given hyperparameter variables;

In [82]:
# Question 2-6-1: Complete the code in the Transformer class based on the defined Encoder and Decoder classes;
class Transformer(nn.Cell):
    def __init__(self, encoder, decoder):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder
        
    def construct(self, enc_inputs, dec_inputs, src_pad_idx, trg_pad_idx):
        """
        enc_inputs: [batch_size, src_len]
        dec_inputs: [batch_size, trg_len]
        """
        enc_outputs, enc_self_attns = self.encoder(enc_inputs, src_pad_idx)

        dec_outputs, dec_self_attns, dec_enc_attns = self.decoder(dec_inputs, enc_inputs, enc_outputs, src_pad_idx, trg_pad_idx)

        dec_logits = dec_outputs.view((-1, dec_outputs.shape[-1]))

        return dec_logits, enc_self_attns, dec_self_attns, dec_enc_attns
        

In [83]:
src_vocab_size = len(de_vocab)
trg_vocab_size = len(en_vocab)
src_pad_idx = de_vocab.pad_idx
trg_pad_idx = en_vocab.pad_idx

d_model = 512
d_ff = 2048
n_layers = 6
n_heads = 8
# Question 2-6-2: Instantiate the transformer model based on the given hyperparameter variables.
encoder = Encoder(src_vocab_size, d_model, n_heads, d_ff, n_layers)
decoder = Decoder(trg_vocab_size, d_model, n_heads, d_ff, n_layers)
model = Transformer(encoder, decoder)

## **Task 3: Training and evaluating a model**

**Subtask 1: Define the loss function and optimizer**

1. Use the MindSpore API to define the loss function. Set the cross-entropy loss function with the ignore_index parameter as trg_pad_idx;

2. Use the MindSpore framework's API to define the optimizer. Use the Adam optimizer with a learning rate of 0.0001;

In [84]:
# Question 3-1-1: Use the MindSpore API to define the loss function. Set the cross-entropy loss function with the ignore_index parameter as trg_pad_idx;
loss_fn = nn.CrossEntropyLoss(ignore_index=trg_pad_idx)
# Question 3-1-2: Use the MindSpore framework's API to define the optimizer. Use the Adam optimizer with a learning rate of 0.0001;

optimizer = nn.Adam(model.trainable_params(), learning_rate=0.0001)

**Subtask 2: Define the forward network computing logic**

**Note:**

During training, the model should predict the &lt;eos&gt; placeholder, not receive it as input. Remove the &lt;eos&gt; placeholder from the end of the target sequence when processing the decoder's input. 

trg = [&lt;bos&gt;, x_1, x_2, ..., x_n, &lt;eos&gt;]

trg[:-1] = [&lt;bos&gt;, x_1, x_2, ..., x_n]


$x_i$ indicates the ith token that represents the content in the target sequence. 

The final output should include &lt;eos&gt; to mark the end of a sentence but not &lt;bos&gt; for the start. When calculating the loss, remove &lt;bos&gt; from the start of the target sequence. 

output = [y_1, y_2, ..., y_n, &lt;eos&gt;]

trg[1:] = [x_1, x_2, ..., x_n, &lt;eos&gt;]

$y_i$ indicates the predicted ith actual content token. 

**Based on the given instructions:**

1. Define the forward function;

2. Define the logical function train_step for a training step;

In [85]:
# Question 3-2-1: Define the forward function;
def forward(enc_inputs, dec_inputs):
    """Forward network
    enc_inputs: [batch_size, src_len]
    dec_inputs: [batch_size, trg_len]
    """
    logits, _, _, _ = model(enc_inputs, dec_inputs[:, :-1], src_pad_idx, trg_pad_idx)

    targets = dec_inputs[:, 1:].view(-1)
    loss = loss_fn(logits, targets.astype(mstype.int32))
    return loss

In [86]:
grad_fn = mindspore.value_and_grad(forward, None, optimizer.parameters)

In [87]:
# Question 3-2-2: Define the logical function train_step fora training step;
def train_step(enc_inputs, dec_inputs):
    loss, grads = grad_fn(enc_inputs, dec_inputs)
    optimizer(grads)
    return loss

**Subtask 3: Define the model training logic train and testing logic evaluate, and train and test the model**

1. Set the model to training mode in the train method. Set the model to non-training mode in the evaluate method;

2. Calculate the loss in the train and evaluate methods;

3. Save the optimal model during training based on valid_loss. The name of the saved model is transformer.ckpt;

In [88]:
from tqdm import tqdm

def train(iterator, epoch=0):
    # Question 3-3-1-1: Set the model to training mode in the train method.
    model.set_train(True)
    num_batches = len(iterator)
    total_loss = 0
    total_steps = 0

    with tqdm(total=num_batches) as t:
        t.set_description(f'Epoch: {epoch}')
        for src, src_len, trg in iterator():
            # Question 3-3-2-1: Calculate the loss in the train method;
            loss = train_step(src, trg)
            total_loss += loss.asnumpy()
            total_steps += 1
            curr_loss = total_loss / total_steps
            t.set_postfix({'loss': f'{curr_loss:.2f}'})
            t.update(1)

    return total_loss / total_steps

In [89]:
def evaluate(iterator):
    # Question 3-3-1-2: Set the model to non-training mode in the evaluate method;
    model.set_train(False)
    num_batches = len(iterator)
    total_loss = 0
    total_steps = 0

    with tqdm(total=num_batches) as t:
        for src, _, trg in iterator():
            # Question 3-3-2-2: Calculate the loss in the evaluate method;
            loss = forward(src, trg)
            total_loss += loss.asnumpy()
            total_steps += 1
            curr_loss = total_loss / total_steps
            t.set_postfix({'loss': f'{curr_loss:.2f}'})
            t.update(1)

    return total_loss / total_steps

**Model Training and Testing**

In [90]:
from mindspore import save_checkpoint

num_epochs = 3
best_valid_loss = float('inf')
ckpt_file_name = './transformer.ckpt'


for i in range(num_epochs):
    train_loss = train(train_iterator, i)
    valid_loss = evaluate(valid_iterator)
    # Question 3-3-3: Save the optimal model during training based on valid_loss. The name of the saved model is transformer.ckpt;
    if valid_loss < best_valid_loss:
        best_valid_loss = valid_loss
        save_checkpoint(model, ckpt_file_name)
        print(f'Epoch {i}: saved model with valid_loss={valid_loss:.4f}')


100%|██████████| 1/1 [00:02<00:00,  2.25s/it, loss=5.56]


Epoch 0: saved model with valid_loss=5.5571


100%|██████████| 1/1 [00:01<00:00,  1.67s/it, loss=4.88]


Epoch 1: saved model with valid_loss=4.8813


100%|██████████| 1/1 [00:01<00:00,  1.64s/it, loss=4.45]


Epoch 2: saved model with valid_loss=4.4548


## **Task 4: Performing model inference**

**Subtask 1: Load model weights to the network**

1. Instantiate the encoder, decoder, and transformer models;

2. Load the trained checkpoint weight parameters to the transformer network structure;

In [91]:
from mindspore import load_checkpoint, load_param_into_net
# Question 4-1-1: Instantiate the encoder, decoder, and transformer models;
encoder = Encoder(src_vocab_size, d_model, n_heads, d_ff, n_layers)
decoder = Decoder(trg_vocab_size, d_model, n_heads, d_ff, n_layers)
new_model = Transformer(encoder, decoder)
# Question 4-1-2: Load the trained checkpoint weight parameters to the transformer network structure;
param_dict = load_checkpoint(ckpt_file_name)
load_param_into_net(new_model, param_dict)

([], [])

**Subtask 2: Implement the inference process**

1. Set the model status to non-training mode in the inference method;

2. Tokenize the input sentence, change all letters to lowercase, and save the result in a list;

3. Add the start and end placeholders and unify the sequence length;

4. Send the input to the encoder and obtain the encoded semantic information;

5. Update dec_inputs;

6. If an end placeholder appears, end the loop;


In [105]:
def inference(sentence, max_len=32):
    """Model inference: Input a German sentence and output the translated English sentence.
    enc_inputs: [batch_size(1), src_len]
    """
    # Question 4-2-1: Set the model to non-training mode in the inference method;
    new_model.set_train(False)
    
    # Question 4-2-2: Tokenize the input sentence, change all letters to lowercase, and save the result in a list;
    if isinstance(sentence, str):
         tokens = re.findall(r'\w+|[^\w\s]', sentence.lower())
    else:
        tokens = [token.lower() for token in sentence]
        
    # Question 4-2-3: Add the start and end placeholders and unify the sequence length;
    # Add code.
    
    tokens = ['<bos>'] + tokens + ['<eos>']
    if len(tokens) < max_len:
        tokens = tokens + ['<pad>'] * (max_len - len(tokens))
    else:
        tokens = tokens[:max_len]
    # enc_inputs: [1, src_len]
    indexes = de_vocab.encode(tokens)
    enc_inputs = Tensor(indexes, mstype.float32).expand_dims(0)
    
    # Question 4-2-4: Send the input to the encoder and obtain the encoded semantic information;
    enc_outputs, _ = new_model.encoder(enc_inputs, src_pad_idx)

    dec_inputs = Tensor([[en_vocab.bos_idx]], mstype.float32)
    
    # dec_inputs: [1, 1]
    max_len = enc_inputs.shape[1]
    for _ in range(max_len):
        dec_outputs, _, _ = new_model.decoder(dec_inputs, enc_inputs, enc_outputs, src_pad_idx, trg_pad_idx)
        dec_logits = dec_outputs.view((-1, dec_outputs.shape[-1]))
        
        dec_logits = dec_logits[-1, :]
        pred = dec_logits.argmax(axis=0).expand_dims(0).expand_dims(0)
        pred = pred.astype(mstype.float32)
        # Question 4-2-5: Update dec_inputs on axis=1;
        dec_inputs = ops.concat([dec_inputs, pred], axis=1)
        # Question 4-2-6: If <eos> appears, end the loop.
        if int(pred.asnumpy()) == en_vocab.eos_idx:
            break
        

    trg_indexes = [int(i) for i in dec_inputs.view(-1).asnumpy()]
    eos_idx = trg_indexes.index(en_vocab.eos_idx) if en_vocab.eos_idx in trg_indexes else -1
    trg_tokens = en_vocab.decode(trg_indexes[1:eos_idx])

    return trg_tokens

Take the first group of sentences in the test dataset as an example. 

In [106]:
example_idx = 0

src = test_dataset[example_idx][0]
trg = test_dataset[example_idx][1]
pred_trg = inference(src)

print(f'src = {src}')
print(f'trg = {trg}')
print(f"predicted trg = {pred_trg}")

src = ['ein', 'mann', 'mit', 'einem', 'orangefarbenen', 'hut', ',', 'der', 'etwas', 'anstarrt', '.']
trg = ['a', 'man', 'in', 'an', 'orange', 'hat', 'starring', 'at', 'something', '.']
predicted trg = ['a', 'man', 'in', 'a', '<unk>', '<unk>', '.']


In [103]:
de_vocab, en_vocab = build_vocab(train_dataset)
print('Unique tokens in de vocabulary:', len(de_vocab))

Unique tokens in de vocabulary: 1288


## BLEU Score

The Bilingual Evaluation Understudy (BLEU) measures how well a text translation model works. It checks how similar the machine's translation $\text{pred}$ is to a human translation $\text{label}$. Each segment's score is calculated by comparing the machine translation to the reference translation. The scores are then added together with weights. The basic rules are:

1. Shorter translations are penalized more;
2. Longer, accurate paragraphs are weighted higher. A full match in a long paragraph means the machine translation is almost the same as the reference;

The BLEU formula is as follows:

$$exp(min(0, 1-\frac{len(\text{label})}{len(\text{pred})})\Pi^k_{n=1}p_n^{1/2^n})$$

- `len(label)`: Human translation length
- `len(pred)`: Machine translation length
- `p_n`: n-gram precision

In [109]:
from nltk.translate.bleu_score import corpus_bleu

def calculate_bleu(dataset, max_len=50):
    trgs = []
    pred_trgs = []
    
    for data in dataset[:100]:
        
        src = data[0]
        trg = data[1]

        pred_trg = inference(src, max_len)
        pred_trgs.append(pred_trg)
        trgs.append([trg])
        
    return corpus_bleu(trgs, pred_trgs)

bleu_score = calculate_bleu(test_dataset)

print(f'BLEU score = {bleu_score*100:.2f}')

BLEU score = 2.63


## Task 5: Optimizing the model
**Subtask 1: Optimize the model**

**Requirements:**

a. Adjust the hyperparameters so that the model's BLEU score on the test dataset reaches over 20;

In [128]:
!/home/ma-user/anaconda3/envs/MindSpore/bin/python /home/ma-user/work/train_transformer.py

  File "/home/ma-user/work/train_transformer.py", line 1
    %%writefile /home/ma-user/work/train_transformer.py
    ^
SyntaxError: invalid syntax


In [124]:
!/home/ma-user/anaconda3/envs/MindSpore/bin/python train_transformer.py

Traceback (most recent call last):
  File "train_transformer.py", line 5, in <module>
    import re, os, mindspore
ModuleNotFoundError: No module named 'mindspore'
